# Visualization 9: Interactive State Contamination Map
## Choropleth Map: Listeria Contamination Rates by State

**Purpose:** Create interactive choropleth map showing state-level contamination patterns

**Data Source:** `usda_fsis_data_product_establishment_specific_laboratory_sampling_rte_product_fy2025.json`

**Output:** Interactive Plotly map showing contamination rates by state with hover tooltips

**Key Features:**
- Color intensity = Contamination rate
- Hover: State details (samples, positives, facilities)
- Exportable as HTML for dashboard integration

---
## Section 1: Setup and Imports

In [ ]:
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
import json
from datetime import datetime

print("Libraries imported successfully!")
print(f"Plotly version: {px.__version__}")
print(f"Analysis date: {datetime.now().strftime('%Y-%m-%d')}")

---
## Section 2: Load and Aggregate Contamination Data by State

In [ ]:
# Load FSIS contamination data
json_path = 'usda_fsis_data_product_establishment_specific_laboratory_sampling_rte_product_fy2025.json'

with open(json_path, 'r') as f:
    fsis_data = json.load(f)[0]

df = pd.DataFrame(fsis_data['data']['primary_table_data'])
print(f"✓ Data loaded: {len(df):,} samples")

In [ ]:
# Filter for Listeria tests only
df_listeria = df[df['lm_listeria_analysis'].notna()].copy()
df_listeria['is_positive'] = df_listeria['lm_listeria_analysis'] == 'Positive'

print(f"Listeria test results: {len(df_listeria):,}")
print(f"Positive samples: {df_listeria['is_positive'].sum():,}")
print(f"Overall contamination rate: {df_listeria['is_positive'].mean() * 100:.2f}%")

In [ ]:
# Aggregate by state
state_stats = df_listeria.groupby('establishment_state').agg({
    'form_id': 'count',
    'is_positive': 'sum',
    'establishment_id': 'nunique'
}).reset_index()

state_stats.columns = ['state', 'total_samples', 'positive_samples', 'num_facilities']
state_stats['contamination_rate'] = (
    state_stats['positive_samples'] / state_stats['total_samples'] * 100
)

# Filter states with at least 50 samples for statistical validity
state_stats = state_stats[state_stats['total_samples'] >= 50].copy()
state_stats = state_stats.sort_values('contamination_rate', ascending=False)

print(f"\n✓ States with ≥50 samples: {len(state_stats)}")
print(f"\nTop 10 states by contamination rate:")
print(state_stats.head(10)[['state', 'total_samples', 'positive_samples', 'contamination_rate']].to_string(index=False))

---
## Section 3: Create Interactive Choropleth Map

In [ ]:
# Create choropleth map
fig = go.Figure(data=go.Choropleth(
    locations=state_stats['state'],
    z=state_stats['contamination_rate'],
    locationmode='USA-states',
    colorscale=[
        [0, 'rgb(255,255,255)'],      # White (0%)
        [0.3, 'rgb(255,220,220)'],    # Light pink
        [0.5, 'rgb(255,150,150)'],    # Pink
        [0.7, 'rgb(255,100,100)'],    # Light red
        [1, 'rgb(200,0,0)']           # Dark red (3%+)
    ],
    zmin=0,
    zmax=3.5,
    colorbar=dict(
        title="Contamination<br>Rate (%)",
        thickness=20,
        len=0.7,
        x=0.92
    ),
    hovertemplate=(
        '<b>%{location}</b><br>' +
        'Contamination Rate: %{z:.2f}%<br>' +
        'Total Samples: %{customdata[0]}<br>' +
        'Positive Samples: %{customdata[1]}<br>' +
        'Facilities Tested: %{customdata[2]}<br>' +
        '<extra></extra>'
    ),
    customdata=state_stats[['total_samples', 'positive_samples', 'num_facilities']].values
))

# Update layout
fig.update_layout(
    title={
        'text': (
            'Listeria Contamination Rate by State (FY2025)<br>' +
            '<sub>USDA FSIS Lab Sampling: Ready-to-Eat Products & Facilities | ' +
            'States with ≥50 samples shown</sub>'
        ),
        'x': 0.5,
        'xanchor': 'center',
        'font': {'size': 20, 'color': '#333', 'family': 'Arial Black'}
    },
    geo=dict(
        scope='usa',
        projection=go.layout.geo.Projection(type='albers usa'),
        showlakes=True,
        lakecolor='rgb(200, 220, 255)',
        bgcolor='rgba(240, 240, 240, 0.5)'
    ),
    height=700,
    font=dict(size=12, family='Arial'),
    margin=dict(l=0, r=0, t=100, b=50)
)

fig.show()
print("\n✓ Interactive map created")

In [ ]:
# Save as HTML for embedding in dashboard
output_html = 'state_contamination_map.html'
fig.write_html(output_html)
print(f"✓ Interactive map saved: {output_html}")
print("  This file can be embedded in a web dashboard or opened in a browser")

---
## Section 4: Create State Statistics Table

In [ ]:
# Create interactive table
fig_table = go.Figure(data=[go.Table(
    header=dict(
        values=[
            '<b>Rank</b>',
            '<b>State</b>',
            '<b>Total Samples</b>',
            '<b>Positive Samples</b>',
            '<b>Contamination Rate (%)</b>',
            '<b>Facilities Tested</b>'
        ],
        fill_color='#4CAF50',
        align='center',
        font=dict(color='white', size=13, family='Arial Bold'),
        height=40
    ),
    cells=dict(
        values=[
            list(range(1, len(state_stats) + 1)),
            state_stats['state'],
            state_stats['total_samples'],
            state_stats['positive_samples'],
            state_stats['contamination_rate'].round(2),
            state_stats['num_facilities']
        ],
        fill_color=[
            ['#f9f9f9' if i % 2 == 0 else 'white' for i in range(len(state_stats))]
        ] * 6,
        align='center',
        font=dict(size=12),
        height=35
    )
)])

fig_table.update_layout(
    title={
        'text': 'State Contamination Statistics (Ranked by Rate)',
        'x': 0.5,
        'xanchor': 'center',
        'font': {'size': 18, 'color': '#333'}
    },
    height=800,
    margin=dict(l=20, r=20, t=80, b=20)
)

fig_table.show()
print("\n✓ State statistics table created")

In [ ]:
# Save table as HTML
output_table_html = 'state_contamination_table.html'
fig_table.write_html(output_table_html)
print(f"✓ State table saved: {output_table_html}")

---
## Section 5: Create Bar Chart - Top 15 States

In [ ]:
# Top 15 states bar chart
top_15 = state_stats.head(15).copy()

fig_bar = go.Figure(data=[
    go.Bar(
        x=top_15['contamination_rate'],
        y=top_15['state'],
        orientation='h',
        marker=dict(
            color=top_15['contamination_rate'],
            colorscale='Reds',
            showscale=True,
            colorbar=dict(title='Rate (%)')
        ),
        text=top_15['contamination_rate'].round(2),
        textposition='outside',
        texttemplate='%{text}%',
        hovertemplate=(
            '<b>%{y}</b><br>' +
            'Rate: %{x:.2f}%<br>' +
            '<extra></extra>'
        )
    )
])

fig_bar.update_layout(
    title={
        'text': 'Top 15 States by Listeria Contamination Rate (FY2025)',
        'x': 0.5,
        'xanchor': 'center',
        'font': {'size': 18, 'color': '#333'}
    },
    xaxis_title='Contamination Rate (%)',
    yaxis_title='State',
    height=600,
    font=dict(size=12),
    yaxis=dict(autorange='reversed'),  # Highest at top
    margin=dict(l=100, r=100, t=100, b=80)
)

fig_bar.show()
print("\n✓ Top 15 states bar chart created")

In [ ]:
# Save bar chart
output_bar_html = 'state_contamination_bar_chart.html'
fig_bar.write_html(output_bar_html)
print(f"✓ Bar chart saved: {output_bar_html}")

---
## Section 6: Export State Data

In [ ]:
# Export state statistics as CSV
output_csv = 'state_contamination_statistics.csv'
state_stats.to_csv(output_csv, index=False)
print(f"✓ State statistics exported: {output_csv}")
print(f"  Records: {len(state_stats)}")
print(f"  Columns: {list(state_stats.columns)}")

In [ ]:
# Create summary statistics JSON
summary = {
    'analysis_date': datetime.now().strftime('%Y-%m-%d'),
    'data_source': 'USDA FSIS Lab Sampling FY2025',
    'fiscal_year': 'FY2025 (Oct 2024 - Sep 2025)',
    'national_stats': {
        'total_samples': int(df_listeria['form_id'].count()),
        'positive_samples': int(df_listeria['is_positive'].sum()),
        'contamination_rate_pct': float(df_listeria['is_positive'].mean() * 100),
        'states_with_data': len(state_stats),
        'total_facilities': int(df_listeria['establishment_id'].nunique())
    },
    'top_5_states': [
        {
            'state': row['state'],
            'contamination_rate': float(row['contamination_rate']),
            'total_samples': int(row['total_samples']),
            'positive_samples': int(row['positive_samples']),
            'facilities': int(row['num_facilities'])
        }
        for _, row in state_stats.head(5).iterrows()
    ],
    'lowest_5_states': [
        {
            'state': row['state'],
            'contamination_rate': float(row['contamination_rate']),
            'total_samples': int(row['total_samples']),
            'positive_samples': int(row['positive_samples']),
            'facilities': int(row['num_facilities'])
        }
        for _, row in state_stats.tail(5).iterrows()
    ]
}

output_json = 'state_contamination_summary.json'
with open(output_json, 'w') as f:
    json.dump(summary, f, indent=2)

print(f"\n✓ Summary statistics exported: {output_json}")
print("\nNational Summary:")
print(json.dumps(summary['national_stats'], indent=2))

---
## Section 7: Summary and Key Findings

In [ ]:
print("="*80)
print("ANALYSIS COMPLETE: State Contamination Map")
print("="*80)

print("\n📊 VISUALIZATIONS CREATED:")
print("   1. state_contamination_map.html - Interactive choropleth map")
print("   2. state_contamination_table.html - Interactive state statistics table")
print("   3. state_contamination_bar_chart.html - Top 15 states bar chart")

print("\n💾 DATA EXPORTED:")
print("   1. state_contamination_statistics.csv - State-level statistics")
print("   2. state_contamination_summary.json - Summary statistics")

print("\n🔍 KEY FINDINGS:")
print(f"\n1. NATIONAL STATISTICS (FY2025):")
print(f"   • Total samples tested: {summary['national_stats']['total_samples']:,}")
print(f"   • Positive samples: {summary['national_stats']['positive_samples']:,}")
print(f"   • National contamination rate: {summary['national_stats']['contamination_rate_pct']:.2f}%")
print(f"   • States with sufficient data (≥50 samples): {summary['national_stats']['states_with_data']}")
print(f"   • Facilities tested: {summary['national_stats']['total_facilities']:,}")

print("\n2. HIGHEST CONTAMINATION STATES:")
for i, state_info in enumerate(summary['top_5_states'], 1):
    print(f"   {i}. {state_info['state']:5s} {state_info['contamination_rate']:5.2f}% "
          f"({state_info['positive_samples']}/{state_info['total_samples']} samples, "
          f"{state_info['facilities']} facilities)")

print("\n3. LOWEST CONTAMINATION STATES:")
for i, state_info in enumerate(summary['lowest_5_states'], 1):
    print(f"   {i}. {state_info['state']:5s} {state_info['contamination_rate']:5.2f}% "
          f"({state_info['positive_samples']}/{state_info['total_samples']} samples, "
          f"{state_info['facilities']} facilities)")

print("\n4. GEOGRAPHIC PATTERNS:")
print("   • Contamination rates vary significantly by state (0% - 3%+)")
print("   • Some states show higher rates despite similar facility counts")
print("   • Regional patterns may exist (requires further analysis)")

print("\n🌐 DASHBOARD INTEGRATION:")
print("   All HTML files can be embedded in a web dashboard using iframes:")
print("   <iframe src='state_contamination_map.html' width='100%' height='700px'></iframe>")

print("\n✅ All visualizations complete!")
print("   Ready for integration into interactive dashboard")
print("="*80)